# 🤖 GPT Smart Assistant with Code Execution


**قابلیت‌ها:**
- تشخیص اینکه سوال نیاز به محاسبات ریاضی داره یا نه
- تولید کد پایتون و اجرای خودکار اون
- retry خودکار در صورت خطا (تا ۳ بار)
- تولید پاسخ نهایی انسانی و طبیعی

## ۱. نصب و import کتابخانه‌ها

In [5]:
# !pip install openai dotenv
from dotenv import load_dotenv
import io
import sys
import os
from openai import OpenAI

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("BASE_URL")

## ۲. تنظیم API Key

In [ ]:
import aisuite as ai

client = ai.Client(
    {
        "openai": {
            "api_key": openai_api_key,
            "base_url": openai_base_url,
        }
    }
)

#MODEL = "openai:gpt-5-nano"
MODEL = "openai:gpt-4o-mini"

## ۳. توابع اصلی

In [ ]:
def check_if_needs_math_update(prompt: str) -> bool:
    """Return True when answering requires any arithmetic or calculation."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """
                            Classify whether answering the question requires any numerical,
                            mathematical, statistical, or logical calculation.

                            Answer YES when the user must calculate something, even when:
                            - the calculation is simple
                            - it can be done mentally
                            - Python is not strictly necessary

                            Return exactly YES or NO.
                            """.strip(),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        max_tokens=5,
        temperature=0,
    )

    answer = response.choices[0].message.content.strip().upper()
    print("Classifier output:", repr(answer))

    return answer == "YES"

def check_if_needs_math(prompt: str) -> bool:
    """Ask GPT whether this question requires mathematical calculation or not."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "Only answer with YES or NO. Do not add any extra explanation."
            },
            {
                "role": "user",
                "content": f"Does the following question require mathematical calculations or executing Python code?\n\nQuestion: {prompt}"
            }
        ],
        max_tokens=10,
        temperature=0
    )
    answer = response.choices[0].message.content.strip().upper()
    return "YES" in answer


def execute_code_safely(code: str):
    """Safely execute Python code and return the output."""
    old_stdout = sys.stdout
    sys.stdout = buffer = io.StringIO()

    # Remove Markdown code fences from the code.
    clean_code = code.strip()
    for marker in ["```python", "```Python", "```", "python"]:
        clean_code = clean_code.replace(marker, "")
    clean_code = clean_code.strip()

    error = None
    try:
        exec(clean_code, {})
    except Exception as e:
        error = str(e)
    finally:
        sys.stdout = old_stdout

    output = buffer.getvalue()
    return output, error, clean_code


def generate_code(prompt: str, error_context: str = None, broken_code: str = None) -> str:
    """Ask GPT to generate Python code (or fix the error)."""
    if error_context and broken_code:
        user_message = f"""The following code has this error: {error_context}

                            Code:
                            {broken_code}

                            Main question: {prompt}

                            Fix the code."""
    else:
        user_message = prompt + "Write only Python code. Do not provide any explanation. Use print() to display the result."

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a Python programmer. Write only pure Python code, without any explanation or markdown. Always use print() to display the result."
            },
            {"role": "user", "content": user_message}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content


def generate_human_response(original_question: str, code: str, output: str) -> str:
    """Generate a natural, human-like response based on the result of the code execution"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are an intelligent assistant who must respond naturally and humanly, without showing code or technical details."
            },
            {
                "role": "user",
                "content": f"""User question: {original_question}

                Calculation result: {output}

                Based on the result, provide a complete, clear, and human-like response. Only give the final answer."""
            }
        ],
        temperature=0.7
    )
    return response.choices[0].message.content


def generate_text_response(prompt: str) -> str:
    """Generate response with no need to execute code, just a text answer."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an intelligent and helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content


print("✅ تمام توابع آماده‌ست!")

✅ تمام توابع آماده‌ست!


## ۴. تابع اصلی Smart Assistant

In [9]:
def smart_assistant(prompt: str, max_attempts: int = 3, verbose: bool = True):
    """
    Smart assistant that:
    - Detects whether the question requires math or not
    - Generates and executes code if needed
    - Generates a human-like response

    Args:
        prompt: User's question
        max_attempts: Maximum attempts to fix the code
        verbose: Show execution details

    Returns:
        dict: Contains answer, code (if any), and code output
    """
    print(f"\n{'='*60}")
    print(f"❓ Question: {prompt}")
    print('='*60)

    # Step 1: Detect question type
    print("🔍 Analyzing question type...")
    needs_math = check_if_needs_math(prompt)

    if needs_math:
        print("🔢 This question requires mathematical calculations.")

        attempt = 0
        error_context = None
        broken_code = None

        while attempt < max_attempts:
            attempt += 1
            print(f"\n⚙️  Attempt {attempt} of {max_attempts}...")

            # Generate code
            raw_code = generate_code(prompt, error_context, broken_code)

            # Execute code
            output, error, clean_code = execute_code_safely(raw_code)

            if verbose:
                print(f"📝 Generated code:\n{'-'*40}\n{clean_code}\n{'-'*40}")

            if error:
                print(f"⚠️  Error: {error}")
                error_context = error
                broken_code = clean_code
            else:
                print(f"✅ Calculations completed successfully!")
                print(f"📊 Code output: {output.strip()}")

                # Generate human-like response
                print("\n💬 Generating final response...")
                human_answer = generate_human_response(prompt, clean_code, output)

                print(f"\n{'='*60}")
                print("💬 Final answer:")
                print(human_answer)
                print('='*60)

                return {
                    "answer": human_answer,
                    "code": clean_code,
                    "output": output,
                    "attempts": attempt,
                    "type": "math"
                }

        print("❌ Unfortunately, we couldn't solve the problem after several attempts.")
        return {"answer": None, "error": "Max attempts reached", "type": "math"}

    else:
        # Regular question
        print("💬 Responding...")
        answer = generate_text_response(prompt)

        print(f"\n{'='*60}")
        print("💬 Answer:")
        print(answer)
        print('='*60)

        return {
            "answer": answer,
            "code": None,
            "output": None,
            "type": "text"
        }

print("✅ smart_assistant function is ready!")

✅ smart_assistant function is ready!


## ۵. تست با مثال‌های مختلف
Rerun all the examples with GPT4.mini

In [ ]:
# Test 1: Simple math question
result = smart_assistant("What is the square root of 144?")

In [15]:
# Test 2: More complex math question
result = smart_assistant(
    "If a 15% discount is applied to a price of 250,000 tomans, what is the final price?"
)


❓ Question: If a 15% discount is applied to a price of 250,000 tomans, what is the final price?
🔍 Analyzing question type...
💬 Responding...

💬 Answer:
212,500 tomans.

Calculation: 250,000 × (1 − 0.15) = 250,000 × 0.85 = 212,500.


In [16]:
# تست ۳: سوال آماری
result = smart_assistant("میانگین اعداد [23, 45, 12, 67, 89, 34] چقدر است؟")


❓ Question: میانگین اعداد [23, 45, 12, 67, 89, 34] چقدر است؟
🔍 Analyzing question type...
💬 Responding...

💬 Answer:
برای محاسبه میانگین، جمع اعداد را بر تعدادشان تقسیم می‌کنیم.
جمع = 23 + 45 + 12 + 67 + 89 + 34 = 270
تعداد = 6
پس میانگین = 270 / 6 = 45

نتیجه: 45


In [19]:
# Test 4: Non-mathematical question
result = smart_assistant("What is the capital of France?")


❓ Question: What is the capital of France?
🔍 Analyzing question type...
💬 Responding...

💬 Answer:
Paris


In [18]:
# Test 5: Programming question requiring calculation
result = smart_assistant("Calculate the factorial of 10.")


❓ Question: Calculate the factorial of 10.
🔍 Analyzing question type...
💬 Responding...

💬 Answer:
10! = 10 × 9 × 8 × 7 × 6 × 5 × 4 × 3 × 2 × 1 = 3,628,800.
